In [2]:
import subprocess
import atexit
import threading
import requests
import sseclient
import json

In [ ]:
## Llama_cpp run cell
# ./llama/llama-server -m "models/Qwen3-14B-Q5_K_M.gguf" -b 512 --n-gpu-layers 1000 -t 8 -n -1 --reasoning-budget 0 --port 51791 --verbose

url = "http://127.0.0.1:51791/v1/chat/completions"

process = subprocess.Popen([
    "./llama/llama-server",
	"-m", "models/Qwen3-14B-Q5_K_M.gguf",
	"-b", "512",
	"--n-gpu-layers", "1000",
	"-t", "8",
	"-n", "-1",
	"--reasoning-budget", "0",
	"--port", "51791",
	"--verbose"
], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, bufsize=1, text=True, encoding="utf-8", start_new_session=True)

atexit.register(process.terminate)

<bound method Popen.terminate of <Popen: returncode: None args: ['./llama/llama-server', '-m', 'models/Qwen3-...>>

In [ ]:
import random

def gen_syllogism(num):
    ops = [random.choice(['a', 'i', 'e', 'o']) for x in range(num-2)]
    ops.append('a')
    ops.append('a')
    random.shuffle(ops)
    ops.append(random.choice(['a', 'i', 'e', 'o']))
    syl = ""
    for i in range(num):
        if i == 0:
            syl = f'S {ops[0]} M1'
        elif i+1 == num:
            syl += f'\nM{i} {ops[i]} P'
            syl += f'\n---------------\nS {ops[i+1]} P'
        else:
            syl += f'\nM{i} {ops[i]} M{i+1}'
        if i+1 == num:
            syl += f'\n'
    return syl

response = requests.post(url, json={'messages': [{'role': 'system', 'content': 'You are a professor of logic and your job is to evaluate syllogisms. A - All x are y; E - No x are y; I - Some x are y; O - Some x are not y. Check if they\'re logically correct and explain their reasoning. Common checks:\n5 Rules (Classical Method)\n1. Middle Term Distribution (M): A middle term must be distributed (appear as the predicate of a general proposition or the subject of a negative proposition) in at least one premise.\n2. Avoiding Two Negative Premises: A valid conclusion cannot be drawn from two negative premises (e.g., "Some S are not P" and "No S is P").\n3. Negative Consistency: A negative premise implies a negative conclusion (if there is a negative premise, the conclusion must be negative).\n4. Assertion Consistency: Two affirmative premises (e.g., "Every A is B") imply an affirmative conclusion.\n5. Decomposition of Terms in the Conclusion: If a term is distributed in the conclusion, it must also be distributed in the premise from which it originates (e.g., S or P).'}, {'role': 'user', 'content': f'Check this syllogism:\n{gen_syllogism(5)}\nAnd provide a text, non-symbolic version of this syllogism, replacing all S, P and M symbolls with some random names.\n\nProvide a brief, not long of a response!\nDO NOT REFER TO RULES BY THEIR NUMBERS/IDS!'}], 'max_tokens': -1})
reply = response.json()['choices'][0]['message']['content']
if "</think>" in reply:
    reply = reply.split("</think>")[1].strip()
print(reply)

In [ ]:
url = "http://127.0.0.1:51790/v1/chat/completions"
syllogism = 'All birds can fly. All event attendants are birds. Some Humans are event attendants. Some Humans can fly.'
response = requests.post(url, json={'messages': [{'role': 'system', 'content': 'You are a professor of logic and your job is to evaluate syllogisms. A - All x are y; E - No x are y; I - Some x are y; O - Some x are not y. Check if they\'re logically correct and explain their reasoning. Common checks:\n5 Rules (Classical Method)\n1. Middle Term Distribution (M): A middle term must be distributed (appear as the predicate of a general proposition or the subject of a negative proposition) in at least one premise.\n2. Avoiding Two Negative Premises: A valid conclusion cannot be drawn from two negative premises (e.g., "Some S are not P" and "No S is P").\n3. Negative Consistency: A negative premise implies a negative conclusion (if there is a negative premise, the conclusion must be negative).\n4. Assertion Consistency: Two affirmative premises (e.g., "Every A is B") imply an affirmative conclusion.\n5. Decomposition of Terms in the Conclusion: If a term is distributed in the conclusion, it must also be distributed in the premise from which it originates (e.g., S or P). Syllogisms can have more than 2 premises. Sometimes a name can appear using its synonyms. Premises do not have to be in order.'}, {'role': 'user', 'content': f'Check this syllogism:\n{syllogism}\nProvide a brief, not long of a response!\nDO NOT REFER TO RULES BY THEIR NUMBERS/IDS!'}], 'max_tokens': -1})
reply = response.json()['choices'][0]['message']['content']
if "</think>" in reply:
    reply = reply.split("</think>")[1].strip()
print(reply)

The syllogism is valid. The middle term "event attendants" is distributed in the premise "All event attendants are birds" (as subject of an A proposition). The conclusion "Some humans can fly" logically follows from the premises through proper term distribution and consistency in affirmativeness.


In [3]:
url = "http://127.0.0.1:51790/v1/chat/completions"
syllogism = 'All birds can fly. All event attendants are birds. Some Humans are event attendants.'
response = requests.post(url, json={'messages': [{'role': 'system', 'content': 'You are a professor of logic and your job is to evaluate syllogisms. A - All x are y; E - No x are y; I - Some x are y; O - Some x are not y. Check if they\'re logically correct and explain their reasoning. Common checks:\n5 Rules (Classical Method)\n1. Middle Term Distribution (M): A middle term must be distributed (appear as the predicate of a general proposition or the subject of a negative proposition) in at least one premise.\n2. Avoiding Two Negative Premises: A valid conclusion cannot be drawn from two negative premises (e.g., "Some S are not P" and "No S is P").\n3. Negative Consistency: A negative premise implies a negative conclusion (if there is a negative premise, the conclusion must be negative).\n4. Assertion Consistency: Two affirmative premises (e.g., "Every A is B") imply an affirmative conclusion.\n5. Decomposition of Terms in the Conclusion: If a term is distributed in the conclusion, it must also be distributed in the premise from which it originates (e.g., S or P). Syllogisms can have more than 2 premises. Sometimes a name can appear using its synonyms. Premises do not have to be in order.'}, {'role': 'user', 'content': f'Check these syllogism premises and make a conclusion based on them:\n{syllogism}\nProvide a brief, not long of a response!\nDO NOT REFER TO RULES BY THEIR NUMBERS/IDS!'}], 'max_tokens': -1})
reply = response.json()['choices'][0]['message']['content']
if "</think>" in reply:
    reply = reply.split("</think>")[1].strip()
print(reply)

The premises imply that all event attendants can fly (from the first two statements), and since some humans are event attendants, it follows that some humans can fly. The conclusion is logically valid.
